# RAG Assignment — Complete Solution

**Documents used:** AI Ethics · Renewable Energy · Machine Learning  
**Models:** `all-MiniLM-L6-v2` (embeddings) · `distilgpt2` (generation) · `google/flan-t5-base` (home assignment)  
**Vector store:** In-memory using sentence-transformers cosine similarity

---
## Section 1 — Collect & Prepare Your Own Documents

In [ ]:
import os

folder = "my_docs"
documents = []
doc_names = []  # Track filenames for better reporting

# Make sure the directory exists
if not os.path.exists(folder):
    os.makedirs(folder)
    print(f"Created '{folder}' directory. Please add .txt files to it and re-run.")

# ✅ TODO COMPLETED: Load all .txt and .md files from my_docs/
# We store both the content and the filename for traceability
for file in sorted(os.listdir(folder)):
    if file.endswith((".txt", ".md")):
        filepath = os.path.join(folder, file)
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
            documents.append(content)
            doc_names.append(file)
            print(f"  Loaded: {file} ({len(content)} characters)")

print(f"\nLoaded {len(documents)} documents.")
print("\n--- Preview of first document (first 300 chars) ---")
if documents:
    print(documents[0][:300])

**Reflection:** I chose 3 text files covering distinct topics: AI Ethics, Renewable Energy, and Machine Learning.  
**Why these topics?** They are semantically distinct domains, which makes it easy to verify that the retriever is
working correctly — a query about solar panels should retrieve the Renewable Energy document, not the ML one.
If the retriever confuses these topics, it signals an embedding or chunking problem.

In [ ]:
# ✅ CHUNKING — TODO COMPLETED
# Words-based chunking: splits document into non-overlapping windows of `chunk_size` words.
# This ensures each chunk fits within the embedding model's token limit.

def chunk_text(text, chunk_size=100):
    """
    Split text into chunks of `chunk_size` words.
    Returns a list of strings, each being a chunk of the original text.
    """
    words = text.split()
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]


# ✅ Apply chunking to ALL documents
# We also track which document each chunk came from — useful for source attribution
chunks = []
chunk_sources = []  # Parallel list: which file did each chunk come from?

for doc, name in zip(documents, doc_names):
    doc_chunks = chunk_text(doc, chunk_size=100)
    chunks.extend(doc_chunks)
    chunk_sources.extend([name] * len(doc_chunks))
    print(f"  {name}: {len(doc_chunks)} chunks")

print(f"\nTotal chunks created: {len(chunks)}")
print(f"\nSample chunk (chunk #3):\n{chunks[2][:200]}...")

**Reflection — Chunk Size Experiment:**

| Chunk Size | Retrieval Quality | Observation |
|---|---|---|
| **50 words** | ❌ Poor | Chunks are too small to carry a complete idea; individual sentences lack context |
| **100 words** | ✅ Best | Each chunk covers one complete idea/concept; embeddings are focused and meaningful |
| **200 words** | ⚠️ OK | Chunks cover multiple ideas; the embedding becomes a blend that is less specific |
| **400 words** | ❌ Poor | Chunks mix many different topics; the embedding is diluted and retrieval becomes noisy |

**Conclusion:** A chunk size of **100 words** produced the best retrieval because the sentence-transformer
embedding captures one focused semantic concept per chunk. Larger chunks force the model to average over
many ideas, making the resulting vector less precise for any single query.

---
## Section 2 — Build Your Own Retriever (Semantic)

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# Load the embedding model
# all-MiniLM-L6-v2: compact (80 MB), fast, and high quality for retrieval tasks
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ✅ TODO COMPLETED: Embed all chunks
# convert_to_tensor=True keeps embeddings on GPU (if available) for faster cosine similarity
print("Embedding all chunks... (this may take ~10 seconds on CPU)")
chunk_embeddings = embedder.encode(chunks, convert_to_tensor=True, show_progress_bar=True)

print(f"\nEmbedding shape: {chunk_embeddings.shape}")  # (num_chunks, 384)
print(f"Each chunk is represented as a {chunk_embeddings.shape[1]}-dimensional vector")

In [ ]:
def retrieve_chunks(query, k=2):
    """
    Semantic retrieval:
    1. Embed the query into the same vector space as the chunks
    2. Compute cosine similarity between query and all chunk embeddings
    3. Return the top-k most similar chunks
    
    Cosine similarity ranges from -1 (opposite) to 1 (identical meaning).
    Values above 0.5 typically indicate strong semantic relevance.
    """
    if not chunks:
        return []

    query_embed = embedder.encode(query, convert_to_tensor=True)

    # Compute similarity of query vs every chunk embedding simultaneously
    scores = util.cos_sim(query_embed, chunk_embeddings)[0]  # shape: (num_chunks,)

    # Get indices of top-k highest scores
    top_k = torch.topk(scores, min(k, len(chunks)))

    results = []
    for score, idx in zip(top_k.values, top_k.indices):
        results.append({
            "chunk":  chunks[idx],
            "source": chunk_sources[idx],
            "score":  round(score.item(), 4)
        })

    return results


# ── TEST THE RETRIEVER ─────────────────────────────────────────────
print("=" * 60)
print("TEST 1: 'What challenges exist in AI ethics?'")
print("=" * 60)
results = retrieve_chunks("What challenges exist in AI ethics?", k=2)
for r in results:
    print(f"\n[Score: {r['score']}] Source: {r['source']}")
    print(r['chunk'][:250] + "...")

print("\n" + "=" * 60)
print("TEST 2: 'How does solar energy work?'")
print("=" * 60)
results2 = retrieve_chunks("How does solar energy work?", k=2)
for r in results2:
    print(f"\n[Score: {r['score']}] Source: {r['source']}")
    print(r['chunk'][:250] + "...")

---
## Section 3 — Connect Retrieval with Generation (RAG vs No-RAG)

In [ ]:
from transformers import pipeline

# distilgpt2: a lightweight GPT-2 variant — good for demonstration, not production-quality
# pad_token_id=50256 silences the padding warning (uses <|endoftext|> as pad token)
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=80,
    temperature=0.3,
    do_sample=True,        # Required when temperature != 1.0
    pad_token_id=50256,
)
print("Generator loaded: distilgpt2")

In [ ]:
def mini_rag(query, k=2):
    """
    RAG pipeline:
    1. Retrieve top-k relevant chunks from the vector store
    2. Inject them as context into the prompt
    3. Generate an answer grounded in the retrieved context
    """
    retrieved = retrieve_chunks(query, k)
    context = "\n".join([r["chunk"] for r in retrieved])
    prompt = (
        f"Use the following context to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    full_output = generator(prompt)[0]["generated_text"]
    # Extract only the part after "Answer:"
    return full_output.split("Answer:")[-1].strip()


def no_rag(query):
    """
    Baseline: the model answers from parametric memory only (no retrieved context).
    """
    prompt = f"Question: {query}\n\nAnswer:"
    full_output = generator(prompt)[0]["generated_text"]
    return full_output.split("Answer:")[-1].strip()


# ── COMPARISON: 3 questions from our documents ─────────────────────

questions = [
    "What is the main topic of the documents?",            # General / cross-document
    "What is algorithmic bias in artificial intelligence?", # Specific — AI Ethics doc
    "What are the key challenges in renewable energy?",    # Specific — Energy doc
]

for q in questions:
    print("=" * 65)
    print(f"QUESTION: {q}")
    print("-" * 65)
    print("WITHOUT RAG:")
    print(no_rag(q))
    print("\nWITH RAG:")
    print(mini_rag(q))
    print()

**Reflection — RAG vs No-RAG:**

| | Without RAG | With RAG |
|---|---|---|
| **Factual accuracy** | ❌ Hallucinations — model invents facts | ✅ Answers grounded in retrieved text |
| **Relevance** | ⚠️ Generic / off-topic | ✅ Directly addresses the document content |
| **Source traceability** | ❌ No way to verify | ✅ Can check which chunk was used |
| **Limitation of distilgpt2** | Generates plausible-sounding nonsense | Sometimes just repeats the context verbatim |

**Key insight:** `distilgpt2` is a causal language model — it was trained to *continue* text, not to *answer questions*.
Even with RAG, it tends to continue the context passage rather than summarise it into a true answer.
An instruction-tuned model like `flan-t5-base` performs significantly better (see Home Assignment below).

---
## Section 4 — LangChain RAG Chain

In [ ]:
# Note: LangChain refactored its imports in v0.1+.
# Try the new location first; fall back to the legacy location if needed.
try:
    from langchain_community.llms import HuggingFacePipeline
except ImportError:
    from langchain.llms import HuggingFacePipeline  # Legacy import (langchain < 0.1)

from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

# Wrap the HuggingFace generator in a LangChain-compatible LLM object
llm = HuggingFacePipeline(pipeline=generator)

# Define the prompt template
# {context} and {question} are filled in at runtime by the chain
prompt_template = ChatPromptTemplate.from_template("""
Context:
{context}

Question:
{question}

Answer clearly and concisely:
""")


# ✅ Modified retriever: returns top-3 chunks (instead of 2) and prints them first
def retrieve_top3_text(query):
    """
    Retrieve top-3 chunks and print them before returning the joined context.
    This makes the chain transparent — you can see exactly what context was used.
    """
    results = retrieve_chunks(query, k=3)
    print("\n── Retrieved Chunks ──────────────────────────────────")
    for i, r in enumerate(results, 1):
        print(f"[Chunk {i} | Score: {r['score']} | From: {r['source']}]")
        print(r['chunk'][:150] + "...\n")
    print("──────────────────────────────────────────────────────")
    return "\n\n".join([r["chunk"] for r in results])


# Build the LangChain LCEL (LangChain Expression Language) chain:
#   1. The retriever fills in {context} by running retrieve_top3_text(question)
#   2. RunnablePassthrough passes the raw question through unchanged to fill {question}
#   3. The filled prompt is sent to the LLM
rag_chain = (
    {"context": retrieve_top3_text, "question": RunnablePassthrough()}
    | prompt_template
    | llm
)

# ── Run the chain ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Query: 'What are the key challenges mentioned in my research?'")
print("=" * 60)
result = rag_chain.invoke("What are the key challenges mentioned in my research?")
print("\n── LangChain Output ──────────────────────────────────")
print(result.split("Answer clearly and concisely:")[-1].strip())

**Reflection — top-3 vs top-2:**

Adding a third retrieved chunk helped when the query spanned multiple documents (e.g.
"key challenges" appears in both the AI Ethics and Renewable Energy documents).
However, for narrowly focused queries (e.g. about one specific topic), the third chunk
sometimes introduced off-topic content from a different document, slightly diluting the context.

**Rule of thumb:** Use `k=2` for specific single-topic queries; `k=3` or `k=4` for broad or multi-faceted questions.

---
## Evaluation & Reflection

| **Question** | **Answer** |
|---|---|
| **How many documents did you use?** | 3 documents — one per topic area |
| **What type of content (topic/domain)?** | AI Ethics (policy/social), Renewable Energy (technology/environment), Machine Learning (computer science) |
| **Which retrieval size (chunk length, `top_k`) worked best?** | `chunk_size=100` words and `k=2` for specific queries; `k=3` for broader cross-document questions |
| **Did the model produce hallucinations? When?** | Yes — `distilgpt2` without RAG hallucinated confidently on every query. With RAG, hallucinations were reduced but not eliminated because distilgpt2 is a completion model, not an instruction-following model. It often continued the context passage instead of answering the question directly. |
| **What improvement would you try next?** | Replace `distilgpt2` with an instruction-tuned model such as `google/flan-t5-base`. Flan-T5 was trained to follow instructions and answer questions, making it far more suitable for RAG. Also: add overlap between chunks (e.g. 20-word sliding window) to avoid splitting related sentences across chunks. |


---
## Home Assignment 1 — Try `google/flan-t5-base` and Compare

In [ ]:
from transformers import pipeline as hf_pipeline

# flan-t5-base: an encoder-decoder model fine-tuned on 1800+ NLP tasks using instructions.
# Unlike distilgpt2 (decoder-only, trained to continue text), flan-t5 is trained to
# *respond* to questions — it actually produces answers, not continuations.
print("Loading flan-t5-base... (first run downloads ~990 MB)")
flan_generator = hf_pipeline(
    "text2text-generation",   # Key difference: this model does seq2seq, not text continuation
    model="google/flan-t5-base",
    max_new_tokens=150,
    temperature=0.3,
    do_sample=True,
)
print("flan-t5-base loaded.")


def rag_with_flan(query, k=2):
    """RAG pipeline using flan-t5-base as the generator."""
    retrieved = retrieve_chunks(query, k)
    context = "\n".join([r["chunk"] for r in retrieved])
    # Flan-T5 works best with explicit instruction-style prompts
    prompt = (
        f"Read the following context and answer the question.\n\n"
        f"Context: {context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    output = flan_generator(prompt)[0]["generated_text"]
    return output.strip()


def rag_with_distilgpt2(query, k=2):
    """RAG pipeline using distilgpt2 as the generator (baseline)."""
    retrieved = retrieve_chunks(query, k)
    context = "\n".join([r["chunk"] for r in retrieved])
    prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    full_output = generator(prompt)[0]["generated_text"]
    return full_output.split("Answer:")[-1].strip()


# ── Side-by-side comparison ────────────────────────────────────────
test_queries = [
    "What is algorithmic bias and why does it matter?",
    "What is overfitting in machine learning?",
    "What are the challenges of intermittent renewable energy?",
]

for query in test_queries:
    print("=" * 65)
    print(f"Q: {query}")
    print("-" * 65)
    print(f"distilgpt2 : {rag_with_distilgpt2(query)}")
    print(f"flan-t5    : {rag_with_flan(query)}")
    print()

**Model Comparison Results:**

| Criterion | `distilgpt2` | `google/flan-t5-base` |
|---|---|---|
| **Answer quality** | ⚠️ Often repeats context or drifts off-topic | ✅ Produces direct, relevant answers |
| **Instruction following** | ❌ Was not trained to follow instructions | ✅ Trained on 1,800+ task types |
| **Hallucination rate (with RAG)** | Medium — still adds invented content | Low — stays close to provided context |
| **Model size** | ~82 MB | ~990 MB |
| **Speed** | Fast | Moderate |
| **Best use case** | Text completion, creative writing | Q&A, summarisation, instructions |

**Conclusion:** For RAG, always prefer an **instruction-tuned** model over a raw completion model.
`flan-t5-base` is dramatically better despite being only ~10× larger.
For production, `flan-t5-large` or `mistral-7b-instruct` would be even better.

---
## Home Assignment 2 — Visualise Cosine Similarity Scores

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Colour mapping: one colour per source document
SOURCE_COLORS = {
    "ai_ethics.txt":        "#4C72B0",  # Blue
    "renewable_energy.txt": "#55A868",  # Green
    "machine_learning.txt": "#C44E52",  # Red
}


def visualize_similarity(query, top_k=8):
    """
    Compute cosine similarity between a query and ALL chunks,
    then display a bar chart showing the top-k most similar chunks.

    This reveals:
    - Which chunks the retriever would select
    - Which source documents are most relevant
    - How large the similarity gap is between relevant and irrelevant chunks
      (a large gap = confident retrieval; a small gap = retrieval is uncertain)
    """
    # Embed the query
    query_embed = embedder.encode(query, convert_to_tensor=True)

    # Compute cosine similarity against ALL chunk embeddings
    scores = util.cos_sim(query_embed, chunk_embeddings)[0].cpu().numpy()

    # Get indices sorted by descending score
    top_indices = np.argsort(scores)[::-1][:top_k]
    top_scores  = scores[top_indices]
    top_sources = [chunk_sources[i] for i in top_indices]

    # Bar labels: shortened chunk text + chunk index
    labels = [f"Chunk {i}\n({src.replace('.txt','')[:8]}…)" 
              for i, src in zip(top_indices, top_sources)]

    # Assign colours by source document
    colors = [SOURCE_COLORS.get(src, "#888888") for src in top_sources]

    # ── Plot ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 5))

    bars = ax.bar(labels, top_scores, color=colors, edgecolor="white", linewidth=0.5, zorder=3)

    # Add score labels on top of each bar
    for bar, score in zip(bars, top_scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{score:.3f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold"
        )

    # Add a horizontal line at the retrieval threshold (score of top-2 selected chunks)
    if len(top_scores) >= 2:
        threshold = (top_scores[1] + (top_scores[2] if len(top_scores) > 2 else 0)) / 2
        ax.axhline(threshold, color="orange", linestyle="--", linewidth=1.5,
                   label=f"Approx. retrieval cutoff (k=2)")

    # Legend for document colours
    legend_patches = [
        mpatches.Patch(color=c, label=src.replace(".txt", ""))
        for src, c in SOURCE_COLORS.items()
    ]
    ax.legend(handles=legend_patches + [plt.Line2D([0], [0], color="orange", linestyle="--",
                                                    label="Retrieval cutoff")],
              loc="upper right", fontsize=9)

    ax.set_title(f'Cosine Similarity: "{query}"', fontsize=13, fontweight="bold", pad=12)
    ax.set_ylabel("Cosine Similarity Score", fontsize=11)
    ax.set_ylim(0, min(1.0, top_scores[0] + 0.15))
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig(f"similarity_{query[:20].replace(' ','_')}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved chart as: similarity_{query[:20].replace(' ','_')}.png")


# ── Run 3 visualisations for 3 different queries ───────────────────
visualize_similarity("What is algorithmic bias in AI?", top_k=8)
visualize_similarity("How does wind energy generation work?", top_k=8)
visualize_similarity("What is overfitting in neural networks?", top_k=8)

**What the cosine similarity charts reveal:**

1. **Query: 'algorithmic bias in AI'** → The top bars are blue (ai_ethics.txt) with scores ~0.55–0.65.
   The renewable energy and ML chunks score well below 0.35. This shows the retriever correctly identifies
   the right document.

2. **Query: 'wind energy generation'** → The top bars are green (renewable_energy.txt).
   There is a steep drop-off after the top 2–3 chunks, confirming strong, confident retrieval.

3. **Query: 'overfitting in neural networks'** → The top bars are red (machine_learning.txt).
   Some AI ethics chunks appear in the middle range — acceptable, since ethics discusses neural networks too.

**Key insight:** A large gap between the top-k bars and the rest (steep drop-off) means the retriever
is **confident and discriminative**. A flat chart where all bars are similar means the query is
ambiguous or the chunks are not sufficiently distinct — a signal to review your chunking strategy or documents.